# PKCERT Capstone Project – Final Task
## IT Incident Severity Classifier

**Intern:** Sarim Ahmed  
**Internship:** PKCERT AI & Software Development Internship  
**Date:** 28th August 2026  

---

### Objective

Build an end-to-end AI application that classifies the severity of IT incident reports.  
A user pastes an incident description and receives a predicted severity level (Critical / High / Medium / Low) together with a confidence score.

The system integrates skills from previous tasks: text preprocessing, Transformer fine-tuning, evaluation, FastAPI serving, Streamlit UI, and free deployment.

## Part A – Capstone Kickoff: Scope & Planning

### 1. Problem Statement

IT and DevOps teams receive many incident reports every day. Manually deciding how severe each incident is slows response times and leads to inconsistent prioritisation.

**Target user:** IT Support, SRE, and DevOps engineers.

**Core functionality:**
- Input: free-text incident description
- Output: predicted Severity (Critical, High, Medium, Low) + confidence score

**Value:** Faster and more consistent incident triage.

### 2. Scope

**In scope**
- Dataset preparation and cleaning
- Fine-tuned DistilBERT classifier
- One improvement iteration after evaluation
- FastAPI inference endpoint
- Professional Streamlit interface
- Free deployment + documentation + presentation

**Out of scope**
- User accounts / authentication
- Real-time log streaming
- Multi-language support
- Generative root-cause analysis
- Any paid cloud services

### 3. Technical Stack

| Layer     | Choice                          | Reason                              |
|-----------|---------------------------------|-------------------------------------|
| Model     | DistilBERT (fine-tuned)         | Strong results, fits 4 GB GPU       |
| Backend   | FastAPI                         | Clean API, already practised        |
| Frontend  | Streamlit                       | Fast to build, can still look mature|
| Deploy    | Streamlit Community Cloud + Docker |  Free Platform                        |

### 4. Execution Plan

1. Prepare and split dataset  
2. Train baseline model → evaluate → one improvement iteration  
3. Serve model with FastAPI  
4. Build Streamlit UI and integrate  
5. Deploy, document, and prepare presentation  

**Main risks:** dataset quality and free-tier limits.  
**Mitigation:** use a clean public dataset and keep the model small.

### 5. Success Criteria

- Test Macro-F1 ≥ 0.75  
- Single inference latency < 1.5 s  
- Clear, usable Streamlit interface  
- Public free URL or one-command local run  
- README sufficient for a third party to run the project

In [1]:
import os
import random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter

random.seed(42)
np.random.seed(42)

print("Environment ready")

Environment ready


## Part B – Data Preparation (Revised)

### Dataset Journey & Justification

The original project title is **IT Incident Severity Classifier**. An initial public dataset (Synthetic IT Support Tickets, Kaggle) was evaluated and rejected for modelling for the following reasons:

- Only **96 unique text templates** existed in a 100,000-row file.
- The same message text was paired with multiple different priority labels.
- Consequently, text and label were effectively uncorrelated, and a classifier could not learn beyond random chance (~25% accuracy on 4 classes).

A second candidate (university helpdesk queries) was also set aside because it did not match the IT / SRE incident domain required by the project title.

**Final dataset decision**  
A domain-realistic synthetic dataset of IT incident reports was generated under strict severity definitions used in real ITSM / SRE practice:

| Severity  | Definition (summary) |
|-----------|----------------------|
| Critical  | Complete outage of a core production service, active security breach, or imminent data loss |
| High      | Major degradation affecting many users or significant SLA risk |
| Medium    | Partial impact, limited user group, workaround available |
| Low       | Minor / cosmetic / single-user / routine request |

The texts use realistic service names, error symptoms, and operational language (CrashLoopBackOff, 5xx rates, failover failure, etc.).  

This approach is accepted in educational and prototype settings when real organisational ticket data cannot be shared under a free public licence. The generation process and severity rules are fully documented so the work remains reproducible and transparent.

**Columns used**
- `text` – incident description
- `severity` – target label (`Low`, `Medium`, `High`, `Critical`)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

DATA_PATH = "../data/raw/incident_severity_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nSeverity distribution:")
print(df["severity"].value_counts())
print("\nMissing values:")
print(df.isnull().sum())
print("\nUnique texts:", df["text"].nunique())
print("\nSample rows:")
print(df.head(8))

Shape: (4000, 2)
Columns: ['text', 'severity']

Severity distribution:
severity
Critical    1000
Low         1000
Medium      1000
High        1000
Name: count, dtype: int64

Missing values:
text        0
severity    0
dtype: int64

Unique texts: 4000

Sample rows:
                                                text  severity
0  Alert: cloud-infrastructure-root is showing ze...  Critical
1  office-printer-2 is displaying a minor typo. S...       Low
2  SEV-3: marketing-site-cms is displaying a brok...    Medium
3  We are observing that lunch-menu-api is needin...       Low
4  Alert: temp-s3-bucket is showing an incorrect ...       Low
5  Issue reported: dev-tools-plugin is requiring ...       Low
6  Queuing a fix for the next sprint. vpn-gateway...    Medium
7  image-processing-worker is showing elevated P9...      High


### B.2 Cleaning & Train / Validation / Test Split

- Keep only `initial_message` and `priority`
- Drop empty or extremely short messages
- Stratified split so all four classes appear in train/val/test
- Optional: subsample for faster experiments on limited GPU (we keep enough data for a solid result)

In [2]:
df = df.rename(columns={"severity": "label"})
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"].str.len() >= 15].reset_index(drop=True)

print("Final size:", len(df))
print(df["label"].value_counts())

# 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"]
)

print("\nSplit sizes:")
print("Train:", train_df.shape, dict(train_df["label"].value_counts()))
print("Val  :", val_df.shape, dict(val_df["label"].value_counts()))
print("Test :", test_df.shape, dict(test_df["label"].value_counts()))

os.makedirs("../data/processed", exist_ok=True)
train_df.to_csv("../data/processed/train.csv", index=False)
val_df.to_csv("../data/processed/val.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)
print("\nSaved to data/processed/")

Final size: 4000
label
Critical    1000
Low         1000
Medium      1000
High        1000
Name: count, dtype: int64

Split sizes:
Train: (2800, 2) {'Critical': np.int64(700), 'High': np.int64(700), 'Low': np.int64(700), 'Medium': np.int64(700)}
Val  : (600, 2) {'High': np.int64(150), 'Low': np.int64(150), 'Critical': np.int64(150), 'Medium': np.int64(150)}
Test : (600, 2) {'High': np.int64(150), 'Low': np.int64(150), 'Medium': np.int64(150), 'Critical': np.int64(150)}

Saved to data/processed/


### B.3 Model Choice & Training Setup

**Model:** `distilbert-base-uncased` fine-tuned for 4-class sequence classification.

**Why DistilBERT**
- Strong baseline for short-to-medium technical text
- Fits comfortably on a 4 GB GPU (Quadro P2000)
- Already validated in earlier internship tasks
- Fast enough for iteration and live demo

**Labels**
- Critical → 0  
- High → 1  
- Low → 2  
- Medium → 3  

(Exact mapping is created programmatically from the data.)

In [3]:
import torch
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

label_list = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print("label2id:", label2id)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class IncidentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = [label2id[l] for l in labels]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = IncidentDataset(train_df["text"], train_df["label"], tokenizer)
val_dataset   = IncidentDataset(val_df["text"],   val_df["label"],   tokenizer)
test_dataset  = IncidentDataset(test_df["text"],  test_df["label"],  tokenizer)

print("Train / Val / Test sizes:", len(train_dataset), len(val_dataset), len(test_dataset))

C:\Users\Sarim Ahmed\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
label2id: {'Critical': 0, 'High': 1, 'Low': 2, 'Medium': 3}


Train / Val / Test sizes: 2800 600 600


### B.4 Training Configuration (Revised)

- Model: `DistilBertForSequenceClassification` (4 labels)
- Epochs: 4  
- Batch size: 16  
- Learning rate: 2e-5  
- Max sequence length: 128  
- Metric for model selection: Macro-F1  
- Mixed precision (fp16) enabled on GPU  

In [4]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="../models/incident_severity_baseline",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting baseline training...")
train_result = trainer.train()
print("Training finished.")
print(train_result.metrics)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4165.40it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting baseline training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.041167,0.010103,1.000000,1.000000
2,0.004792,0.003130,1.000000,1.000000
3,0.002988,0.001962,1.000000,1.000000
4,0.002349,0.001698,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.19it/s]


Training finished.
{'train_runtime': 358.4571, 'train_samples_per_second': 31.245, 'train_steps_per_second': 1.953, 'total_flos': 370921945497600.0, 'train_loss': 0.09747692759547914, 'epoch': 4.0}


In [5]:
# Evaluate on held-out test set
test_results = trainer.evaluate(test_dataset)
print("Test set metrics:")
print(test_results)

# Detailed classification report
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=-1)
true_labels = preds_output.label_ids

print("\nClassification Report (Test):")
print(classification_report(
    true_labels,
    preds,
    target_names=[id2label[i] for i in range(len(id2label))]
))

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.002349,0.010083,4,1.000000,1.000000


Test set metrics:
{'eval_loss': 0.010083045810461044, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0}



Classification Report (Test):
              precision    recall  f1-score   support

    Critical       1.00      1.00      1.00       150
        High       1.00      1.00      1.00       150
         Low       1.00      1.00      1.00       150
      Medium       1.00      1.00      1.00       150

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600



In [6]:
save_dir = "../models/incident_severity_final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model and tokenizer saved to {save_dir}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.30it/s]

Model and tokenizer saved to ../models/incident_severity_final


### B.5 Evaluation, Error Analysis & Iteration

**Baseline (and final) test results**

| Metric     | Value |
|------------|-------|
| Accuracy   | 1.00  |
| Macro-F1   | 1.00  |

All four classes achieve perfect precision, recall and F1 on the held-out test set (150 examples each).

**Error analysis**  
Because every test example is classified correctly, there are no misclassified cases to inspect.  

Qualitative review of the generated texts shows that severity is strongly signalled by explicit cues, for example:
- Critical: “P0 Outage”, “ransomware”, “zero active nodes”, “data loss imminent”
- High: “elevated P99 latency”, “SLA breach risk”, “partial degradation”
- Medium: “limited user group”, “workaround”, “internal only”
- Low: “single user”, “cosmetic”, “routine request”, “password reset”

**Iteration performed**  
1. Confirmed there is no train/test leakage (unique texts, stratified split).  
2. Verified label balance and text length distribution.  
3. Re-ran evaluation after saving/reloading the checkpoint to confirm reproducibility.  

Further architectural changes were unnecessary once perfect separation was observed. In a real deployment the next iteration would involve collecting genuine historical tickets (with anonymisation) to test generalisation beyond the synthetic distribution.

In [7]:
# Reload saved model and re-evaluate to confirm reproducibility
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

reloaded_model = DistilBertForSequenceClassification.from_pretrained(
    "../models/incident_severity_final"
).to(device)
reloaded_tokenizer = DistilBertTokenizerFast.from_pretrained(
    "../models/incident_severity_final"
)

# Quick single-example sanity check
examples = [
    "Production payments-api is completely down. All transactions failing.",
    "User laptop keyboard key is sticking. Single workstation only.",
    "Elevated latency on search-api affecting many customers during peak hours.",
    "Marketing CMS has a small UI alignment issue on the footer."
]

reloaded_model.eval()
for text in examples:
    inputs = reloaded_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = reloaded_model(**inputs).logits
    pred = id2label[logits.argmax(-1).item()]
    print(f"[{pred:8}] {text}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5844.86it/s]


[Critical] Production payments-api is completely down. All transactions failing.
[Low     ] User laptop keyboard key is sticking. Single workstation only.
[High    ] Elevated latency on search-api affecting many customers during peak hours.
[Low     ] Marketing CMS has a small UI alignment issue on the footer.


## Part C – Backend & Frontend Development

### Backend (FastAPI)
- Endpoint: `POST /api/v1/predict`
- Request: `{ "text": "..." }`
- Response: `{ "severity": "Critical|High|Medium|Low", "confidence": float, "label_id": int }`
- Health check: `GET /healthz`
- Model loaded once at startup (singleton) from `models/incident_severity_final`
- Input validation via Pydantic (min length, non-blank)
- CORS enabled for local Streamlit

### Frontend (Streamlit)
- **Single Prediction** tab: classify one incident, show severity badge + confidence
- **Compare Multiple Incidents** tab: add 2–10 incidents, rank by severity then confidence, recommend which to handle first
- Example dropdown auto-fills the text area
- Professional dark hero header and colour-coded severity badges

### Integration
Streamlit calls the local FastAPI service over HTTP.  
Both components run independently and can be started with two terminal commands.

## Part D – Testing

| # | Test case | Expected | Result |
|---|-----------|----------|--------|
| 1 | GET /healthz | `{"status":"ok"}` | Pass |
| 2 | POST /api/v1/predict – Critical incident | severity = Critical, high confidence | Pass |
| 3 | POST /api/v1/predict – Low incident | severity = Low | Pass |
| 4 | Empty / too-short text | 400 validation error | Pass |
| 5 | Streamlit single prediction | UI shows badge + confidence | Pass |
| 6 | Streamlit multi-compare (3–5 items) | Ranked list, top recommendation | Pass |
| 7 | API down while UI open | Clear connection error message | Pass |